# Part 5 · datetime
> datetime / date / timedelta / timezone / dateutil / pytz / pandas datetime

## 1. datetime 基础

In [ ]:
from datetime import datetime, date, time, timedelta, timezone

# --- 创建 ---
datetime(2024, 1, 15, 10, 30, 0)               # 指定时间
datetime.now()                                  # 本地当前时间（无时区信息）
datetime.utcnow()                               # UTC 当前时间（无时区信息，已弃用）
datetime.now(tz=timezone.utc)                  # UTC 当前时间（有时区信息，推荐）
datetime.today()                               # 等价 datetime.now()
datetime.fromtimestamp(1705305600)             # Unix 时间戳 → datetime（本地时区）
datetime.fromtimestamp(1705305600, tz=timezone.utc)  # UTC
datetime.fromisoformat('2024-01-15T10:30:00')  # ISO 字符串 → datetime
datetime.strptime('15/01/2024 10:30', '%d/%m/%Y %H:%M')  # 自定义格式

# --- date ---
date.today()                           # 今天的 date 对象
date(2024, 1, 15)                      # 指定日期
date.fromtimestamp(1705305600)         # 时间戳 → date
date.fromisoformat('2024-01-15')       # ISO 字符串 → date

# --- 属性 ---
dt = datetime(2024, 1, 15, 10, 30, 45, 123456)
dt.year         # 2024
dt.month        # 1
dt.day          # 15
dt.hour         # 10
dt.minute       # 30
dt.second       # 45
dt.microsecond  # 123456
dt.weekday()    # 0=Monday ... 6=Sunday
dt.isoweekday() # 1=Monday ... 7=Sunday
dt.isocalendar() # (year, week_number, weekday)
dt.date()       # 提取 date 部分
dt.time()       # 提取 time 部分
dt.tzinfo       # 时区信息（无则 None）

## 2. 格式化 & 解析

In [ ]:
dt = datetime(2024, 1, 15, 10, 30, 45)

# --- strftime（datetime → 字符串）---
dt.strftime('%Y-%m-%d')            # '2024-01-15'
dt.strftime('%Y/%m/%d %H:%M:%S')  # '2024/01/15 10:30:45'
dt.strftime('%d %b %Y')           # '15 Jan 2024'
dt.strftime('%A, %B %d, %Y')      # 'Monday, January 15, 2024'
dt.strftime('%Y-W%V')             # '2024-W03'  ISO 周数
dt.isoformat()                    # '2024-01-15T10:30:45'  ISO 8601
dt.isoformat(timespec='minutes')  # '2024-01-15T10:30'

# --- 常用格式符 ---
# %Y  4位年   %y  2位年
# %m  月(01-12)  %B  月份全名  %b  月份缩写
# %d  日(01-31)  %j  年中第几天
# %H  时(00-23)  %I  时(01-12)
# %M  分(00-59)  %S  秒(00-59)
# %f  微秒(000000-999999)
# %A  星期全名   %a  星期缩写  %w  星期(0=Sunday)
# %V  ISO周数   %U  周数(Sunday为首)
# %Z  时区名称  %z  UTC偏移量(+HHMM)

# --- strptime（字符串 → datetime）---
datetime.strptime('2024-01-15', '%Y-%m-%d')
datetime.strptime('15/01/2024 10:30', '%d/%m/%Y %H:%M')

# --- dateutil.parser.parse（自动识别格式，更智能）---
from dateutil.parser import parse
parse('January 15, 2024')          # 自动识别
parse('15/01/2024', dayfirst=True)
parse('2024-01-15T10:30:00+08:00') # 带时区

## 3. timedelta — 时间差

In [ ]:
from datetime import timedelta

# --- 创建 ---
timedelta(days=7)
timedelta(days=1, hours=2, minutes=30)
timedelta(seconds=3600)        # 1 小时
timedelta(weeks=2)             # 14 天

# --- 算术 ---
dt = datetime(2024, 1, 15)
dt + timedelta(days=7)         # 7天后
dt - timedelta(days=1)         # 昨天

end   = datetime(2024, 3, 1)
start = datetime(2024, 1, 1)
diff  = end - start            # timedelta 对象
diff.days                      # 60（只有整数天）
diff.seconds                   # 剩余秒数（不含天数部分）
diff.total_seconds()           # 总秒数（5184000.0）
diff.days * 24 + diff.seconds/3600  # 总小时数

# --- relativedelta（月、年操作）---
from dateutil.relativedelta import relativedelta

dt + relativedelta(months=1)           # 下个月同一天
dt + relativedelta(years=1)            # 明年同一天
dt + relativedelta(months=-3)          # 3个月前
dt + relativedelta(month=12, day=31)   # 今年最后一天（绝对替换）

diff = relativedelta(datetime(2024,3,15), datetime(2024,1,1))
diff.months  # 2
diff.days    # 14

# --- 常用场景 ---
# 本月第一天
today = date.today()
month_start = today.replace(day=1)

# 本月最后一天
import calendar
last_day = calendar.monthrange(today.year, today.month)[1]
month_end = today.replace(day=last_day)

# 上周一
today - timedelta(days=today.weekday() + 7)

# N 个工作日后
from dateutil.rrule import rrule, DAILY, MO, TU, WE, TH, FR
from datetime import date
biz_days = list(rrule(DAILY, count=10, byweekday=(MO,TU,WE,TH,FR), dtstart=date.today()))

## 4. 时区处理

In [ ]:
from datetime import datetime, timezone, timedelta

# --- 标准库 timezone ---
utc = timezone.utc
est = timezone(timedelta(hours=-5))   # 手动定义固定偏移

dt_utc = datetime.now(tz=timezone.utc)   # 当前 UTC 时间（aware datetime）
dt_utc.timestamp()                        # → Unix 时间戳

# --- pytz（更完整的时区库）---
import pytz

utc = pytz.utc
ny  = pytz.timezone('America/New_York')
sh  = pytz.timezone('Asia/Shanghai')

# naive → aware（添加时区）
dt_naive = datetime(2024, 1, 15, 10, 0, 0)
dt_aware = ny.localize(dt_naive)           # 正确做法（处理 DST）

# 转换时区
dt_sh = dt_aware.astimezone(sh)            # 纽约时间 → 上海时间
dt_utc = dt_aware.astimezone(pytz.utc)    # → UTC

# aware → naive（去掉时区）
dt_naive = dt_aware.replace(tzinfo=None)

# --- zoneinfo（Python 3.9+，推荐）---
from zoneinfo import ZoneInfo

dt = datetime(2024, 1, 15, 10, 0, tzinfo=ZoneInfo('America/New_York'))
dt.astimezone(ZoneInfo('Asia/Shanghai'))
dt.astimezone(ZoneInfo('UTC'))

# --- 数据工程常见场景 ---
# UTC 存储，展示时转本地时间（推荐做法）
# 1. 所有数据存为 UTC
# 2. 查询时转用户时区

# pandas 时区
import pandas as pd
s = pd.to_datetime(['2024-01-15 10:00:00'])
s = s.dt.tz_localize('UTC')              # 标注为 UTC
s = s.dt.tz_convert('America/New_York')  # 转换时区

## 5. Unix 时间戳 & 常用转换

In [ ]:
from datetime import datetime, timezone

# datetime → Unix 时间戳
dt = datetime(2024, 1, 15, tzinfo=timezone.utc)
ts = dt.timestamp()                    # 1705276800.0  (秒)
ts_ms = int(dt.timestamp() * 1000)    # 毫秒时间戳（JavaScript 常用）

# Unix 时间戳 → datetime
datetime.fromtimestamp(ts, tz=timezone.utc)   # UTC datetime
datetime.fromtimestamp(ts)                     # 本地时间（不推荐）
datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc)  # 毫秒时间戳

# pandas 时间戳
import pandas as pd
pd.Timestamp('2024-01-15')
pd.Timestamp.now()
pd.Timestamp.now(tz='UTC')
pd.Timestamp('2024-01-15').timestamp()  # Unix 秒
pd.to_datetime(1705276800, unit='s')    # 秒 → Timestamp
pd.to_datetime(1705276800000, unit='ms') # 毫秒 → Timestamp

# 速查：格式与函数
# '2024-01-15'                    → datetime.strptime / pd.to_datetime
# '2024-01-15T10:30:00+00:00'     → datetime.fromisoformat / pd.to_datetime
# 1705276800  (Unix 秒)            → datetime.fromtimestamp
# 1705276800000 (Unix 毫秒)        → datetime.fromtimestamp(ts/1000)
# '15/01/2024'                    → strptime('%d/%m/%Y') 或 dateutil.parse